# Import libraries

In [1]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.model_selection import train_test_split
from scipy.stats import entropy
import matplotlib.pyplot as plt
import seaborn as sns
from torch.utils.data import Sampler
import torch.optim as optim
from sklearn.decomposition import PCA
import tensorflow as tf

2026-10-16 13:05:27.041070: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 AVX512F AVX512_VNNI FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-10-16 13:05:27.542160: I tensorflow/core/util/util.cc:169] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-10-16 13:05:27.670187: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcudart.so.11.0'; dlerror: libcudart.so.11.0: cannot open shared object file: No such file or directory
2026-10-16 13:05:27.670211: I tensorflow/stream_executor/cuda/cudart_stub.cc:29] Ignore above cudart dlerror if yo

# Load ESOL dataset

In [2]:
real_data = pd.read_csv("../datasets/ESOL_dataset.csv")
threshold = []
batch_size = 20
# ECFP fingerprints
X = real_data.iloc[:,:-2].values.astype(np.float32)
# Actual solubility values
y = real_data.iloc[:, -2:-1].values.astype(np.float32)

# Encoder

In [3]:
def build_conditional_encoder(
    input_dim: int,
    label_dim: int,
    latent_dim: int,
    hidden_layers=(512,256,128),
    activation="elu",
    use_label_embedding: bool = False,
    label_embed_dim: int = 16
):
    x_in = tf.keras.Input(shape=(input_dim,), name="x")
    y_in = tf.keras.Input(shape=(label_dim,), name="y")

    y_feat = y_in
    if use_label_embedding:
        y_feat = tf.keras.layers.Dense(
            label_embed_dim,
            activation=activation,
            name="y_embed"
        )(y_in)

    h = tf.keras.layers.Concatenate(name="enc_concat")([x_in, y_feat])

    for i, units in enumerate(hidden_layers):
        h = tf.keras.layers.Dense(
            units,
            activation=activation,
            name=f"enc_dense_{i}"
        )(h)

    z_mean = tf.keras.layers.Dense(latent_dim, name="z_mean")(h)
    z_logvar = tf.keras.layers.Dense(latent_dim, name="z_logvar")(h)

    return tf.keras.Model([x_in, y_in], [z_mean, z_logvar], name="cvae_encoder")

# Decoder

In [4]:
def build_conditional_decoder(
    input_dim: int,
    label_dim: int,
    latent_dim: int,
    hidden_layers=(128,256,512),
    activation="gelu",
    output_activation=None,
    use_label_embedding: bool = False,
    label_embed_dim: int = 16,
):
    z_in = tf.keras.Input(shape=(latent_dim,), name="z")
    y_in = tf.keras.Input(shape=(label_dim,), name="y")

    y_feat = y_in
    if use_label_embedding:
        y_feat = tf.keras.layers.Dense(
            label_embed_dim,
            activation=activation,
            name="y_embed"
        )(y_in)

    h = tf.keras.layers.Concatenate(name="dec_concat")([z_in, y_feat])

    for i, units in enumerate(hidden_layers):
        h = tf.keras.layers.Dense(
            units,
            activation=activation,
            name=f"dec_dense_{i}"
        )(h)

    x_logits = tf.keras.layers.Dense(
        input_dim,
        activation=None,
        name="x_logits"
    )(h)

    return tf.keras.Model([z_in, y_in], x_logits, name="cvae_decoder")


# Binary sampling

The decoder outputs logits. During generation, sigmoid converts them to Bernoulli probabilities and each ECFP bit is sampled as exactly 0 or 1.


In [5]:
def sampling(z_mean, z_logvar):
    """
    Reparameterization trick.
    """
    eps = tf.random.normal(tf.shape(z_mean))
    return z_mean + tf.exp(0.5 * z_logvar) * eps

# VAE Model

In [6]:
class ConditionalVAE(tf.keras.Model):
    def __init__(
        self,
        encoder: tf.keras.Model,
        decoder: tf.keras.Model,
        beta: float = 0.1,
        max_beta: float = None,
        warmup_epochs: int = 0,
        recon_loss: str = "bce",
        name="ConditionalVAE",
    ):
        super().__init__(name=name)
        self.encoder = encoder
        self.decoder = decoder

        self.beta = float(beta)
        self.max_beta = float(max_beta) if max_beta is not None else None
        self.warmup_epochs = int(warmup_epochs)
        self.recon_loss = recon_loss.lower()

        self.loss_tracker = tf.keras.metrics.Mean(name="loss")
        self.recon_tracker = tf.keras.metrics.Mean(name="recon")
        self.kl_tracker = tf.keras.metrics.Mean(name="kl")
        self.beta_tracker = tf.keras.metrics.Mean(name="beta")
        self._epoch_var = tf.Variable(0, dtype=tf.int32, trainable=False)

    @property
    def metrics(self):
        return [
            self.loss_tracker,
            self.recon_tracker,
            self.kl_tracker,
            self.beta_tracker,
        ]

    def set_epoch(self, epoch: int):
        self._epoch_var.assign(int(epoch))

    def _current_beta(self):
        if self.warmup_epochs <= 0 or self.max_beta is None:
            return tf.constant(self.beta, dtype=tf.float32)

        e = tf.cast(self._epoch_var, tf.float32)
        w = tf.cast(self.warmup_epochs, tf.float32)
        frac = tf.clip_by_value(e / w, 0.0, 1.0)

        return (
            tf.constant(self.beta, tf.float32)
            + frac * (
                tf.constant(self.max_beta, tf.float32)
                - tf.constant(self.beta, tf.float32)
            )
        )

    def _reconstruction_loss(self, x, x_logits):
        # ECFP fingerprints are binary (0/1), so use Bernoulli likelihood.
        # from_logits=True is more numerically stable than applying sigmoid
        # before BCE.
        per_dim = tf.nn.sigmoid_cross_entropy_with_logits(
            labels=x,
            logits=x_logits
        )

        # Sum over fingerprint bits, then average over the batch.
        return tf.reduce_mean(tf.reduce_sum(per_dim, axis=1))

    def train_step(self, data):
        (x, y), _ = data

        with tf.GradientTape() as tape:
            z_mean, z_logvar = self.encoder([x, y], training=True)
            z = sampling(z_mean, z_logvar)

            # Decoder returns logits.
            x_logits = self.decoder([z, y], training=True)

            recon = self._reconstruction_loss(x, x_logits)

            kl = tf.reduce_mean(
                -0.5 * tf.reduce_sum(
                    1 + z_logvar
                    - tf.square(z_mean)
                    - tf.exp(z_logvar),
                    axis=1,
                )
            )

            beta = self._current_beta()
            loss = recon + beta * kl

        grads = tape.gradient(loss, self.trainable_variables)
        self.optimizer.apply_gradients(
            zip(grads, self.trainable_variables)
        )

        self.loss_tracker.update_state(loss)
        self.recon_tracker.update_state(recon)
        self.kl_tracker.update_state(kl)
        self.beta_tracker.update_state(beta)

        return {m.name: m.result() for m in self.metrics}

    def call(self, inputs, training=False):
        x, y = inputs

        z_mean, z_logvar = self.encoder(
            [x, y],
            training=training
        )

        z = sampling(z_mean, z_logvar)

        # Return logits, consistent with the decoder.
        return self.decoder([z, y], training=training)


class EpochSetter(tf.keras.callbacks.Callback):
    def on_epoch_begin(self, epoch, logs=None):
        if hasattr(self.model, "set_epoch"):
            self.model.set_epoch(epoch)


# Train the model

In [7]:
input_dim = X.shape[1]
latent_dim = 128
label_dim = 1
epochs = 500

encoder = build_conditional_encoder(
    input_dim=input_dim,
    label_dim=label_dim,
    latent_dim=latent_dim,
    hidden_layers=[512,256,128],
    activation="elu",
    use_label_embedding=True,
)

decoder = build_conditional_decoder(
    input_dim=input_dim,
    label_dim=label_dim,
    latent_dim=latent_dim,
    hidden_layers=[128,256,512],
    activation="gelu",
    output_activation=None,
    use_label_embedding=True,
)

model = ConditionalVAE(
    encoder=encoder,
    decoder=decoder,
    beta=1,
    max_beta=1,
    warmup_epochs=80,
    recon_loss="bce",
)

model.compile(
    optimizer=tf.keras.optimizers.Adam(5e-4)
)

# -----------------------------
# Train
# -----------------------------
history = model.fit(
    x=(X,y),
    y=X,
    epochs=epochs,
    batch_size=batch_size,
    shuffle=True,
    callbacks=[EpochSetter()]
)


2026-10-16 13:06:26.184951: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcuda.so.1'; dlerror: libcuda.so.1: cannot open shared object file: No such file or directory
2026-10-16 13:06:26.186497: W tensorflow/stream_executor/cuda/cuda_driver.cc:263] failed call to cuInit: UNKNOWN ERROR (303)
2026-10-16 13:06:26.186573: I tensorflow/stream_executor/cuda/cuda_diagnostics.cc:156] kernel driver does not appear to be running on this host (DESKTOP-LFUOV61): /proc/driver/nvidia/version does not exist
2026-10-16 13:06:26.189067: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 AVX512F AVX512_VNNI FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.


Epoch 1/500
57/57 [==============================] - 2s 10ms/step - loss: 231.3546 - recon: 217.0642 - kl: 14.2904 - beta: 1.0000
Epoch 2/500
57/57 [==============================] - 1s 9ms/step - loss: 86.3156 - recon: 83.5938 - kl: 2.7218 - beta: 1.0000
Epoch 3/500
57/57 [==============================] - 0s 8ms/step - loss: 83.3991 - recon: 81.2690 - kl: 2.1301 - beta: 1.0000
Epoch 4/500
57/57 [==============================] - 0s 8ms/step - loss: 82.5331 - recon: 80.6256 - kl: 1.9076 - beta: 1.0000
Epoch 5/500
57/57 [==============================] - 1s 9ms/step - loss: 81.3588 - recon: 79.6815 - kl: 1.6774 - beta: 1.0000
Epoch 6/500
57/57 [==============================] - 1s 9ms/step - loss: 80.8463 - recon: 79.0971 - kl: 1.7492 - beta: 1.0000
Epoch 7/500
57/57 [==============================] - 1s 9ms/step - loss: 79.8310 - recon: 77.9483 - kl: 1.8827 - beta: 1.0000
Epoch 8/500
57/57 [==============================] - 0s 8ms/step - loss: 78.6676 - recon: 76.4612 - kl: 2.2065 - b

# Binary sampling

The decoder outputs logits. During generation, sigmoid converts them to Bernoulli probabilities and each ECFP bit is sampled as exactly 0 or 1.


In [8]:
target_solubility = np.percentile(y,100)
column_names = real_data1.columns
y_target = np.array([[target_solubility]], dtype=np.float32).astype(np.float32).reshape(-1)

def generate_conditioned(decoder, y_cond, n=1000, temperature=1.0):

    y_cond = tf.convert_to_tensor(
        y_cond,
        dtype=tf.float32
    )

    if len(y_cond.shape) == 1:
        y_cond = tf.repeat(
            y_cond[None, :],
            repeats=n,
            axis=0
        )

    z = tf.random.normal(
        (tf.shape(y_cond)[0], latent_dim)
    )

    # Decoder returns logits
    logits = decoder(
        [z, y_cond],
        training=False
    )

    # Convert logits to Bernoulli probabilities
    probs = tf.sigmoid(logits / temperature)

    # Sample binary ECFP bits
    x_gen = tf.cast(
        tf.random.uniform(tf.shape(probs)) < probs,
        tf.float32
    )

    return x_gen


x_gen = generate_conditioned(
    decoder,
    y_target,
    n=1000,
    temperature=1
)

def postprocess_generated(
    x_gen,
    scaler,
    feature_names,
    log_type="ln_eps",
    eps=1e-8
):
    # tensor -> numpy
    if hasattr(x_gen, "numpy"):
        x_gen = x_gen.numpy()

    # Ensure binary values.
    x_gen = np.rint(x_gen).astype(np.int8)

    return pd.DataFrame(
        x_gen,
        columns=feature_names
    )


df_gen = postprocess_generated(
    x_gen,
    scaler=scaler,
    feature_names=column_names,
    log_type="ln_eps"
)




In [9]:
df_gen.to_csv('../datasets/gen_ESOL_class1_cont.csv')